# DARF v5 — Anisotropic α + Realism Base Option

**v4 öğrendiğimiz:** Daenerys α'sını her blokta düşürmek = realism kaybı + saç rengi kaybı. Trade-off'u yanlış eksende çözdük.

**v5 stratejisi:** Üç ekseni **ayrı ayrı** kontrol et:

| Eksen | Mekanizma |
|---|---|
| **Realism / kalite** | Up-block α'yı YÜKSEK tut (1.40+) → face/hair texture LoRA detaylı kalır |
| **Orientation** | Down/mid α'yı DÜŞÜK tut (0.15/0.45) → pose composition LoRA-free, ControlNet domine eder |
| **Identity hair color** | Up-block + spatial gate floor=0 → leak yok, Daenerys saçı baskın |

**Bonus seçenek:** `BASE_MODEL` flag'i ile **Juggernaut-XL-v9** veya **RealVisXL-V4.0** kullan. Bunlar multi-person realism'de SDXL-base'den çok daha güçlü; cartoon sorunu kökten çözülür.

## Bölüm 1 — Setup

**`BASE_MODEL` seçimi**:
- `"stabilityai/stable-diffusion-xl-base-1.0"` (default, hızlı, indirilmiş, biraz cartoon)
- `"RunDiffusion/Juggernaut-XL-v9"` (önerilir, ~7GB indirme, çok daha realistic)
- `"SG161222/RealVisXL_V4.0"` (alternatif, photorealism odaklı)

In [ ]:
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
# BASE_MODEL = "RunDiffusion/Juggernaut-XL-v9"     # önerilen, realistic
# BASE_MODEL = "SG161222/RealVisXL_V4.0"            # alternatif

In [ ]:
import os
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git {REPO_DIR}
else:
    print("Pulling latest...")
    !cd {REPO_DIR} && git pull

!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)
import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)

os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0" "mediapipe==0.10.14" "protobuf<5" scipy

os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"Base model: {BASE_MODEL}")

## Bölüm 2 — Pipeline + Scorers + Evaluator

In [ ]:
from pipeline import load_identities, build_pipeline
from scorer import FaceScorer
from pose_scorer import PoseScorer
from attribute_scorer import AttributeScorer
from evaluator import Evaluator
from PIL import Image
from IPython.display import display
import pipeline as P

identities = load_identities()

# Front-facing emphasis ama saç rengi token'ları en başta
identities["hermione"]["visual_description"] = (
    "front-facing young woman with long bushy brown curly hair, hazel eyes, "
    "fair skin, looking directly at the camera, wearing a black Hogwarts school robe"
)
identities["daenerys"]["visual_description"] = (
    "front-facing woman with long straight platinum silver-white hair, violet eyes, "
    "pale skin, looking directly at the camera, wearing a regal blue dress"
)

identities["hermione"]["attention_phrases"] = [
    "Hermione_Granger",
    "long bushy brown curly hair",        # ← saç en güçlü identity sinyali
    "front-facing young woman",
    "black Hogwarts school robe",
]
identities["daenerys"]["attention_phrases"] = [
    "dae woman",
    "long straight platinum silver-white hair",   # ← saç ön planda
    "front-facing woman",
    "regal blue dress",
]

identities["hermione"]["negative_features"] = (
    "blonde hair, silver hair, platinum hair, white hair, crown, "
    "side profile, turned head, profile view"
)
identities["daenerys"]["negative_features"] = (
    "brown hair, brunette, bushy hair, curly hair, witch robe, school uniform, "
    "side profile, turned head, profile view"
)

P._BASE_NEGATIVE = (
    "blurry, low quality, deformed face, cartoon, illustration, 3d render, anime, "
    "three women, three people, third person, extra person, woman in background, "
    "side profile, profile view, turned head, looking sideways"
)

pipe = build_pipeline(identities, base_model=BASE_MODEL)

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
attr_scorer = AttributeScorer(identities, device="cuda")
evaluator   = Evaluator(device="cuda")
ref_embeds = {k: evaluator.extract_face_embedding(
    Image.open(identities[k]["reference_image"]).convert("RGB")) for k in identities}

names = list(identities.keys())
identity_regions = {names[0]: (0,0,512,1024), names[1]: (512,0,1024,1024)}

import numpy as np, cv2
def detected_face_count(img):
    img_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    return len(evaluator.face_app.get(img_bgr))
def show_eval(label, image):
    sim_h, sim_d = evaluator.detect_and_assign_faces(image, ref_embeds["hermione"], ref_embeds["daenerys"])
    n = detected_face_count(image)
    print(f"[Eval/{label}] arc_h={sim_h:+.3f}  arc_d={sim_d:+.3f}  faces={n}")

print("\nReady.")

## Bölüm 3 — Pose

In [ ]:
from face_aware_pose import make_face_aware_pose, get_face_aware_target_keypoints
pose_dense = make_face_aware_pose(1024, 1024, front_facing=True, dense_face=True)
target_keypoints = get_face_aware_target_keypoints(front_facing=True)
display(pose_dense.resize((384, 384)))

## Bölüm 4 — v5 Ana: Anisotropic α (DOWN düşük, UP yüksek)

Daenerys için:
- `down=0.15` → composition/orientation LoRA-free
- `mid=0.50` → orta yol
- `up=1.40` → face/hair detail GÜÇLÜ → realism + saç rengi

Hermione için:
- `down=0.10, mid=0.20, up=0.45` → arka planda, dominansı kontrollü

In [ ]:
from pipeline import generate_two_stage

v5_config = dict(
    layout_lora_scales={"hermione": 0.10, "daenerys": 0.30},
    identity_lora_scales={
        "hermione": {"down": 0.10, "mid": 0.20, "up": 0.45},
        "daenerys": {"down": 0.15, "mid": 0.50, "up": 1.40},   # ← anisotropic
    },
    layout_ctrl_scale=1.00,
    identity_ctrl_scale=0.95,                                     # 1.0 yerine 0.95 (refine biraz nefes)
    refine_strength=0.35,                                          # 0.25→0.35 realism için biraz nefes
    refine_steps=28, layout_steps=28,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.20, "mid": 0.05, "up": 0.00},  # up'ta SIFIR leak
    use_bg_lock=True, bg_lock_ratio=0.40, bg_lock_padding=24,
    bg_lock_in_layout=True, bg_lock_in_identity=False,
    identity_regions=identity_regions,
)

stage1, stage2 = generate_two_stage(pipe, identities, pose_dense, seed=42, **v5_config)

print("\n[Stage 1]");  display(stage1.resize((512,512)))
print("[Stage 2]");    display(stage2.resize((512,512)))
show_eval("v5_main_stage1", stage1)
show_eval("v5_main_stage2", stage2)

## Bölüm 5 — Face-Local Refine (saç dahil geniş crop, orta strength)

In [ ]:
from pipeline import face_local_refine

# Strength dengesi: çok düşük → cartoon kalır, çok yüksek → orientation döner
v5_final = face_local_refine(
    pipe, identities, stage2,
    face_scorer, identity_regions,
    refine_strength=0.40,        # 0.28→0.40 realism için
    crop_pad_ratio=0.8,          # saç + boyun dahil ama omuzlardan çok büyük değil
    refine_steps=30,
    seed=42,
    feather=28,
)
print("\n[v5 final]")
display(v5_final.resize((512,512)))
show_eval("v5_final", v5_final)

## Bölüm 6 — Anisotropy Ablation (Daenerys up-block sweep, down/mid sabit)

**Hipotez:** Up-block α arttıkça realism+identity↑ ama (eğer down/mid de yükselirse) orientation↓.
v5'in iddiası: down/mid sabit kalıp sadece up'ı yükseltirsek **orientation bozulmadan identity gelir**.

In [ ]:
anisotropy_sweep = [
    ("d_up_0.8",  {"down": 0.15, "mid": 0.50, "up": 0.80}),
    ("d_up_1.1",  {"down": 0.15, "mid": 0.50, "up": 1.10}),
    ("d_up_1.4",  {"down": 0.15, "mid": 0.50, "up": 1.40}),    # v5 default
    ("d_up_1.7",  {"down": 0.15, "mid": 0.50, "up": 1.70}),
    # Karşılaştırma için ISOtropic (her blok aynı)
    ("d_iso_1.1", {"down": 1.10, "mid": 1.10, "up": 1.10}),
]

aniso_results = {}
for label, dscale in anisotropy_sweep:
    print(f"\n=== {label} {dscale} ===")
    cfg = {**v5_config}
    cfg["identity_lora_scales"] = {
        "hermione": {"down": 0.10, "mid": 0.20, "up": 0.45},
        "daenerys": dscale,
    }
    _, s2 = generate_two_stage(pipe, identities, pose_dense, seed=42, **cfg)
    aniso_results[label] = s2
    display(s2.resize((384, 384)))
    show_eval(label, s2)

## Bölüm 7 — Multi-Seed (rapor için final)

In [ ]:
v5_seed_imgs = {}
v5_seed_finals = {}
for s in [42, 123, 777]:
    print(f"\n=== v5 seed {s} ===")
    _, s2 = generate_two_stage(pipe, identities, pose_dense, seed=s, **v5_config)
    final = face_local_refine(
        pipe, identities, s2, face_scorer, identity_regions,
        refine_strength=0.40, crop_pad_ratio=0.8,
        refine_steps=30, seed=s, feather=28,
    )
    v5_seed_imgs[s] = s2
    v5_seed_finals[s] = final
    print("Stage 2:");  display(s2.resize((384, 384)))
    print("Final:");    display(final.resize((384, 384)))
    show_eval(f"v5_seed{s}_stage2", s2)
    show_eval(f"v5_seed{s}_final",  final)

## Bölüm 8 — Ablation + ZIP

In [ ]:
import pandas as pd

def collect(label, image):
    fc = face_scorer.score_image(image, identity_regions)
    pc = pose_scorer.score_image(image, target_keypoints, identity_regions)
    ac = attr_scorer.score_image(image, identity_regions)
    sim_h, sim_d = evaluator.detect_and_assign_faces(image, ref_embeds["hermione"], ref_embeds["daenerys"])
    n = detected_face_count(image)
    return [
        {"method": label, "identity": k,
         "arcface_self":  round(fc[k]["arcface"], 4),
         "dominance":     round(fc[k]["dominance"], 4),
         "pose":          round(pc[k]["pose"], 4),
         "attr_margin":   round(ac[k]["margin"], 4),
         "eval_arcface":  round(sim_h if k == "hermione" else sim_d, 4),
         "face_count":    n}
        for k in ["hermione", "daenerys"]
    ]

rows = []
if "stage2"   in dir(): rows += collect("v5_main_stage2", stage2)
if "v5_final" in dir(): rows += collect("v5_final",       v5_final)
if "aniso_results" in dir():
    for label, img in aniso_results.items():
        rows += collect(label, img)
if "v5_seed_imgs" in dir():
    for s, img in v5_seed_imgs.items():
        rows += collect(f"v5_seed{s}_stage2", img)
if "v5_seed_finals" in dir():
    for s, img in v5_seed_finals.items():
        rows += collect(f"v5_seed{s}_final", img)

df = pd.DataFrame(rows)
out = "data/results/darf_v5"
os.makedirs(out, exist_ok=True)
df.to_csv(f"{out}/ablation.csv", index=False)
df

In [ ]:
# Save + ZIP
if "stage1" in dir():     stage1.save(f"{out}/v5_main_stage1.png")
if "stage2" in dir():     stage2.save(f"{out}/v5_main_stage2.png")
if "v5_final" in dir():   v5_final.save(f"{out}/v5_final.png")
if "aniso_results" in dir():
    for label, img in aniso_results.items():
        img.save(f"{out}/{label}.png")
if "v5_seed_imgs" in dir():
    for s, img in v5_seed_imgs.items():
        img.save(f"{out}/v5_seed{s}_stage2.png")
if "v5_seed_finals" in dir():
    for s, img in v5_seed_finals.items():
        img.save(f"{out}/v5_seed{s}_final.png")
pose_dense.save(f"{out}/pose_skeleton.png")

import shutil
from IPython.display import FileLink
shutil.make_archive("/workspace/darf_v5_results", "zip", out)
print(f"Zip ready: /workspace/darf_v5_results.zip   (base: {BASE_MODEL})")
FileLink("/workspace/darf_v5_results.zip")

## Akademik Çerçeve

v4 demonstrated that uniformly lowering identity LoRA α suppresses orientation bias — but at the cost of identity faithfulness and image realism. v5 reframes this as a **block-anisotropic α scheduling problem**: identity bias and identity expression live in different UNet block subspaces, and these subspaces can be controlled independently.

**Key v5 contribution:**
$$\alpha_{i,l} = \begin{cases} \alpha_{\text{low}} & \text{if } l \in \text{down/mid (composition)} \\ \alpha_{\text{high}} & \text{if } l \in \text{up (face/hair detail)} \end{cases}$$

Combined with the spatially gated LoRA residual (block-floor: 0.20 in down, 0.0 in up), the up-block carries identity and texture detail with no rival leakage, while down/mid blocks remain LoRA-light so ControlNet's pose conditioning is not overpowered by the LoRA's training-data orientation bias.

**Ablation evidence (Bölüm 6):** isotropic α=1.1 vs anisotropic (down=0.15/mid=0.50/up=1.4) — same average effective α, very different orientation/identity balance.

**Optional realism baseline:** the same DARF stack applied on Juggernaut-XL or RealVisXL eliminates the cartoonish artifacts inherent to SDXL-base, demonstrating that DARF mechanisms are base-model-agnostic.

Final claim:
> *"Multi-LoRA identity expression and orientation bias occupy distinct UNet block subspaces. DARF v5 schedules per-block α anisotropically (low in composition blocks, high in detail blocks) and combines this with a per-block spatial gate floor, decoupling the identity-vs-orientation trade-off that uniform scaling cannot resolve."*